# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [9]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [161]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv(".env")

# API Keys
def require_api_key(value: str | None) -> str:
    """Return a non-empty API key or raise a configuration error."""
    if value is None or not value.strip():
        raise ValueError(
            "require_api_key | ValueError | Configuration: OPENAI_API_KEY | "
            "API key is missing. Set OPENAI_API_KEY in .env or the environment."
        )
    return value.strip()


OPENAI_API_KEY = require_api_key(os.getenv("OPENAI_API_KEY"))

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Load email credentials without requiring them for meal generation.
EMAIL_SENDER = os.getenv("EMAIL_SENDER", "").strip()
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [162]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [58]:
for missing_value in (None, "", "   "):
    try:
        require_api_key(missing_value)
    except ValueError as error:
        message = str(error)

        assert "require_api_key" in message
        assert "ValueError" in message
        assert "Configuration: OPENAI_API_KEY" in message
        assert "Set OPENAI_API_KEY" in message
    else:
        raise AssertionError("Missing API key was unexpectedly accepted.")

assert require_api_key(" example-test-value ") == "example-test-value"

print(message)
print("All API key configuration checks passed.")

require_api_key | ValueError | Configuration: OPENAI_API_KEY | API key is missing. Set OPENAI_API_KEY in .env or the environment.
All API key configuration checks passed.


In [59]:
import os
from unittest.mock import patch

# Simulate an environment without an API key.
with patch.dict(os.environ, {}, clear=True):
    try:
        require_api_key(os.getenv("OPENAI_API_KEY"))
    except ValueError as error:
        message = str(error)

        assert "require_api_key" in message
        assert "Configuration: OPENAI_API_KEY" in message
        assert "Set OPENAI_API_KEY" in message

        print(message)
    else:
        raise AssertionError("Missing API key was unexpectedly accepted.")

print("Missing environment key check passed.")

require_api_key | ValueError | Configuration: OPENAI_API_KEY | API key is missing. Set OPENAI_API_KEY in .env or the environment.
Missing environment key check passed.


## 3. Pydantic Models for Validation

In [163]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"

class MealPlanRequest(BaseModel):
    """Validate a single meal-plan generation request."""
    dietary_preference: DietaryPreference
    theme_requests: str | None = "random"
    additional_preferences: str | None = "none"

class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        expected_count = len(DAYS)

        if len(v) != expected_count:
            raise ValueError(
                f"Meal plan must have exactly {expected_count} recipes, "
                f"got {len(v)}"
            )

        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All recipe themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [164]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

In [165]:
import json


def parse_model_json(content: str) -> dict:
    """Parse plain JSON or a complete outer JSON code fence."""
    text = content.strip()

    if text.startswith("```"):
        lines = text.splitlines()

        if (
            len(lines) < 3
            or lines[0].strip() not in ("```json", "```")
            or lines[-1].strip() != "```"
        ):
            raise ValueError(
                "Expected a complete outer code fence labeled json or unlabeled."
            )

        text = "\n".join(lines[1:-1])

    data = json.loads(text)

    if not isinstance(data, dict):
        raise ValueError("The model response must contain a JSON object.")

    return data

In [63]:
assert parse_model_json('{"recipes": []}') == {"recipes": []}
assert parse_model_json('  {"recipes": []}  ') == {"recipes": []}

try:
    parse_model_json('{"recipes": [}')
except json.JSONDecodeError:
    print("Malformed JSON was rejected.")
else:
    raise AssertionError("Malformed JSON was unexpectedly accepted.")

try:
    parse_model_json('["unexpected", "list"]')
except ValueError as error:
    assert "JSON object" in str(error)
    print("A non-object JSON value was rejected.")
else:
    raise AssertionError("A JSON list was unexpectedly accepted.")

print("Plain JSON parsing checks passed.")

Malformed JSON was rejected.
A non-object JSON value was rejected.
Plain JSON parsing checks passed.


In [65]:
# Accept complete outer fences.
assert parse_model_json(
    '```json\n{"recipes": []}\n```'
) == {"recipes": []}

assert parse_model_json(
    '```\n{"recipes": []}\n```'
) == {"recipes": []}

assert parse_model_json(
    '  ```json\n{"recipes": []}\n```  '
) == {"recipes": []}

# Reject unsupported wrappers and surrounding text.
unsupported_responses = [
    '```json\n{"recipes": []}',
    '```python\n{"recipes": []}\n```',
    'Here is your plan:\n```json\n{"recipes": []}\n```',
    '```json\n{"recipes": []}\n```\nAdditional text',
]

for response in unsupported_responses:
    try:
        parse_model_json(response)
    except ValueError:
        pass
    else:
        raise AssertionError("An unsupported response was unexpectedly accepted.")

print("All JSON code fence checks passed.")

All JSON code fence checks passed.


In [166]:
def validate_meal_plan(data: dict) -> MealPlan:
    """Validate parsed data against the meal-plan schema."""
    return MealPlan.model_validate(data)

In [78]:
import json
from copy import deepcopy
from pathlib import Path
from pydantic import ValidationError

# Load saved data without calling the API.
validation_fixture = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

validated_plan = validate_meal_plan(validation_fixture)

assert isinstance(validated_plan, MealPlan)
assert validated_plan.model_dump() == validation_fixture

# Create an independent invalid copy.
invalid_fixture = deepcopy(validation_fixture)
invalid_fixture["recipes"] = invalid_fixture["recipes"][:6]

try:
    validate_meal_plan(invalid_fixture)
except ValidationError as error:
    assert any(
        issue["loc"] == ("recipes",)
        for issue in error.errors()
    )
    print("Invalid recipe count was rejected.")
else:
    raise AssertionError("An invalid meal plan was unexpectedly accepted.")

print("All meal-plan validation helper checks passed.")

Invalid recipe count was rejected.
All meal-plan validation helper checks passed.


In [167]:
def make_failure_result(error_message: str, generation_time: float) -> dict:
    """Build a consistent failed generation result."""
    return {
        "success": False,
        "meal_plan": None,
        "generation_time": round(generation_time, 2),
        "error": error_message,
    }

In [87]:
failure_result = make_failure_result(
    error_message="Simulated failure",
    generation_time=1.234,
)

assert failure_result == {
    "success": False,
    "meal_plan": None,
    "generation_time": 1.23,
    "error": "Simulated failure",
}

assert make_failure_result("Another failure", 0)["generation_time"] == 0

print(failure_result)
print("All failure result helper checks passed.")

{'success': False, 'meal_plan': None, 'generation_time': 1.23, 'error': 'Simulated failure'}
All failure result helper checks passed.


In [168]:
def invoke_meal_plan_chain(chain, request_data: dict):
    """Invoke the meal-plan chain with prepared request data."""
    return chain.invoke(request_data)

In [104]:
from unittest.mock import Mock

# Verify request forwarding and response preservation.
fake_chain = Mock()
fake_response = object()
fake_chain.invoke.return_value = fake_response

sample_request = {
    "dietary_preference": "omnivore",
    "theme_requests": "random",
    "additional_preferences": "none",
    "format_instructions": "Test schema instructions",
}

response = invoke_meal_plan_chain(fake_chain, sample_request)

fake_chain.invoke.assert_called_once_with(sample_request)
assert response is fake_response

# Verify that invocation errors reach the caller.
failing_chain = Mock()
simulated_error = RuntimeError("Simulated invocation failure")
failing_chain.invoke.side_effect = simulated_error

try:
    invoke_meal_plan_chain(failing_chain, sample_request)
except RuntimeError as error:
    assert error is simulated_error
else:
    raise AssertionError("The invocation error was unexpectedly suppressed.")

failing_chain.invoke.assert_called_once_with(sample_request)

print("All chain invocation helper checks passed.")

All chain invocation helper checks passed.


In [169]:
def build_meal_plan_request(
    dietary_preference: str,
    theme_requests: str | None,
    additional_preferences: str | None,
    format_instructions: str,
) -> dict:
    """Build normalized input for the meal-plan chain."""
    return {
        "dietary_preference": dietary_preference,
        "theme_requests": normalize_optional_text(
            theme_requests, "random diverse cuisines"
        ),
        "additional_preferences": normalize_optional_text(
            additional_preferences, "none"
        ),
        "format_instructions": format_instructions,
    }

In [107]:
# Verify normalization and preservation of request fields.
request = build_meal_plan_request(
    dietary_preference="vegan",
    theme_requests=" Italian ",
    additional_preferences=" no nuts ",
    format_instructions="Test schema",
)

assert request == {
    "dietary_preference": "vegan",
    "theme_requests": "Italian",
    "additional_preferences": "no nuts",
    "format_instructions": "Test schema",
}

# Verify defaults for missing and blank optional text.
for empty_value in (None, "", "   "):
    request = build_meal_plan_request(
        dietary_preference="omnivore",
        theme_requests=empty_value,
        additional_preferences=empty_value,
        format_instructions="Test schema",
    )

    assert request["theme_requests"] == "random diverse cuisines"
    assert request["additional_preferences"] == "none"

print(request)
print("All request preparation checks passed.")

{'dietary_preference': 'omnivore', 'theme_requests': 'random diverse cuisines', 'additional_preferences': 'none', 'format_instructions': 'Test schema'}
All request preparation checks passed.


## 5. Meal Generation Logic

In [170]:
import json
import time

from pydantic import ValidationError

from openai import (
    AuthenticationError,
    RateLimitError,
    APITimeoutError,
    APIConnectionError,
    APIStatusError,
)

class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)
        self.shopping_parser = JsonOutputParser(pydantic_object=ShoppingItem)

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        recipe_chain = MEAL_PLAN_PROMPT | self.llm

        try:
            request_data = build_meal_plan_request(
                dietary_preference=dietary_preference,
                theme_requests=theme_requests,
                additional_preferences=additional_preferences,
                format_instructions=self.recipe_parser.get_format_instructions(),
            )

            result = invoke_meal_plan_chain(recipe_chain, request_data)

            meal_plan_data = parse_model_json(result.content)
            meal_plan = validate_meal_plan(meal_plan_data)
            generation_time = time.time() - start_time

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(generation_time, 2),
                "error": None
            }

        except AuthenticationError as e:
            return make_failure_result(
                error_message=(
                    "MealPlanGenerator.generate | AuthenticationError | "
                    f"recipe_chain.invoke | HTTP {e.status_code} | "
                    "Authentication failed. Check that OPENAI_API_KEY is valid "
                    "and active, then reload the configuration."
                ),
                generation_time=time.time() - start_time,
            )

        except RateLimitError as e:
            return make_failure_result(
                error_message=(
                    "MealPlanGenerator.generate | RateLimitError | "
                    f"recipe_chain.invoke | HTTP {e.status_code} | "
                    "Request rejected due to rate or quota limits. "
                    "Check API quota and billing; if requests are too frequent, "
                    "wait before trying again."
                ),
                generation_time=time.time() - start_time,
            )

        except APITimeoutError:
            return make_failure_result(
                error_message=(
                    "MealPlanGenerator.generate | APITimeoutError | "
                    "recipe_chain.invoke | Request timed out. "
                    "Check your connection and try again later."
                ),
                generation_time=time.time() - start_time,
            )

        except APIConnectionError:
            return make_failure_result(
                error_message=(
                    "MealPlanGenerator.generate | APIConnectionError | "
                    "recipe_chain.invoke | Could not connect to the API. "
                    "Check your network, proxy, and firewall settings."
                ),
                generation_time=time.time() - start_time,
            )

        except APIStatusError as e:
            return make_failure_result(
                error_message=(
                    f"MealPlanGenerator.generate | {type(e).__name__} | "
                    f"recipe_chain.invoke | HTTP {e.status_code} | "
                    "The API rejected the request. "
                    "For 5xx errors, check service availability and try later; "
                    "for other errors, check request parameters and model access."
                ),
                generation_time=time.time() - start_time,
            )

        except json.JSONDecodeError as e:
            return make_failure_result(
                error_message=(
                    "MealPlanGenerator.generate | JSONDecodeError | "
                    f"Response JSON parsing: line {e.lineno}, "
                    f"column {e.colno}, position {e.pos} | "
                    "The model response contains invalid JSON. "
                    "Check JSON syntax, quotation marks, commas, and brackets."
                ),
                generation_time=time.time() - start_time,
            )

        except ValidationError as e:
            field_errors = []

            for issue in e.errors():
                location = ".".join(str(part) for part in issue["loc"])
                location = location or "meal_plan"
                field_errors.append(f"{location}: {issue['type']}")

            return make_failure_result(
                error_message=(
                    "MealPlanGenerator.generate | ValidationError | "
                    f"MealPlan schema: {'; '.join(field_errors)} | "
                    "The response does not match the meal-plan requirements. "
                    "Check the listed fields, required values, recipe count, "
                    "ingredient count, and unique themes."
                ),
                generation_time=time.time() - start_time,
            )

        except ValueError:
            return make_failure_result(
                error_message=(
                    "MealPlanGenerator.generate | ValueError | "
                    "Response processing | Unsupported response format. "
                    "Check that the response contains a JSON object, either "
                    "plain or inside one complete outer json or unlabeled code fence."
                ),
                generation_time=time.time() - start_time,
            )

        except Exception as e:
            return make_failure_result(
                error_message=(
                    f"MealPlanGenerator.generate | {type(e).__name__} | "
                    "Generation pipeline | An unexpected failure occurred. "
                    "Check the generation steps and reproduce the failure "
                    "with an isolated offline test."
                ),
                generation_time=time.time() - start_time,
            )


# Initialize generator
generator = MealPlanGenerator()

In [100]:
from unittest.mock import patch

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

# Inject an unexpected failure without calling the API.
with patch.object(
    chain_type,
    "invoke",
    side_effect=RuntimeError("PRIVATE_RUNTIME_MARKER"),
) as invoke_mock:
    test_result = generator.generate(
        dietary_preference="omnivore",
        theme_requests="random",
        additional_preferences="none",
    )

    invoke_mock.assert_called_once()

assert test_result["success"] is False
assert test_result["meal_plan"] is None
assert isinstance(test_result["generation_time"], (int, float))

message = test_result["error"]

assert "MealPlanGenerator.generate" in message
assert "RuntimeError" in message
assert "Generation pipeline" in message
assert "Check the generation steps" in message
assert "PRIVATE_RUNTIME_MARKER" not in message

print(message)
print("Unexpected error handling check passed.")

MealPlanGenerator.generate | RuntimeError | Generation pipeline | An unexpected failure occurred. Check the generation steps and reproduce the failure with an isolated offline test.
Unexpected error handling check passed.


In [88]:
import httpx
from openai import AuthenticationError
from unittest.mock import patch

# Construct a simulated HTTP response without sending a request.
fake_request = httpx.Request(
    "POST",
    "https://api.openai.com/v1/chat/completions",
)

fake_response = httpx.Response(
    status_code=401,
    request=fake_request,
)

# Include a marker to verify that raw exception text is not exposed.
private_marker = "PRIVATE_TEST_MARKER"

fake_error = AuthenticationError(
    message=f"Invalid credentials: {private_marker}",
    response=fake_response,
    body={"error": {"message": "Simulated authentication failure"}},
)

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

with patch.object(
    chain_type,
    "invoke",
    side_effect=fake_error,
) as invoke_mock:
    auth_result = generator.generate(
        dietary_preference="omnivore",
        theme_requests="random",
        additional_preferences="none",
    )

    invoke_mock.assert_called_once()

assert auth_result["success"] is False
assert auth_result["meal_plan"] is None
assert isinstance(auth_result["generation_time"], (int, float))

error_message = auth_result["error"]

assert "MealPlanGenerator.generate" in error_message
assert "AuthenticationError" in error_message
assert "recipe_chain.invoke" in error_message
assert "HTTP 401" in error_message
assert "Check that OPENAI_API_KEY is valid" in error_message
assert private_marker not in error_message

print(error_message)
print("Authentication error handling check passed.")

MealPlanGenerator.generate | AuthenticationError | recipe_chain.invoke | HTTP 401 | Authentication failed. Check that OPENAI_API_KEY is valid and active, then reload the configuration.
Authentication error handling check passed.


In [90]:
import httpx
from openai import RateLimitError
from unittest.mock import patch

# Construct a simulated response without sending a request.
rate_response = httpx.Response(
    status_code=429,
    request=httpx.Request(
        "POST",
        "https://api.openai.com/v1/chat/completions",
    ),
)

rate_error = RateLimitError(
    message="Simulated rate limit: PRIVATE_RATE_MARKER",
    response=rate_response,
    body={"error": {"message": "Simulated rate limit"}},
)

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

with patch.object(
    chain_type,
    "invoke",
    side_effect=rate_error,
) as invoke_mock:
    rate_result = generator.generate(
        dietary_preference="omnivore",
        theme_requests="random",
        additional_preferences="none",
    )

    invoke_mock.assert_called_once()

assert rate_result["success"] is False
assert rate_result["meal_plan"] is None
assert isinstance(rate_result["generation_time"], (int, float))

message = rate_result["error"]

assert "MealPlanGenerator.generate" in message
assert "RateLimitError" in message
assert "recipe_chain.invoke" in message
assert "HTTP 429" in message
assert "Check API quota and billing" in message
assert "PRIVATE_RATE_MARKER" not in message

print(message)
print("Rate limit handling check passed.")

MealPlanGenerator.generate | RateLimitError | recipe_chain.invoke | HTTP 429 | Request rejected due to rate or quota limits. Check API quota and billing; if requests are too frequent, wait before trying again.
Rate limit handling check passed.


In [92]:
import httpx
from openai import APITimeoutError, APIConnectionError
from unittest.mock import patch

# Create request metadata without sending a network request.
network_request = httpx.Request(
    "POST",
    "https://api.openai.com/v1/chat/completions",
)

network_cases = [
    (
        APITimeoutError(request=network_request),
        "APITimeoutError",
        "Check your connection",
    ),
    (
        APIConnectionError(
            message="PRIVATE_CONNECTION_MARKER",
            request=network_request,
        ),
        "APIConnectionError",
        "Check your network",
    ),
]

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

for simulated_error, expected_type, expected_suggestion in network_cases:
    with patch.object(
        chain_type,
        "invoke",
        side_effect=simulated_error,
    ) as invoke_mock:
        network_result = generator.generate(
            dietary_preference="omnivore",
            theme_requests="random",
            additional_preferences="none",
        )

        invoke_mock.assert_called_once()

    assert network_result["success"] is False
    assert network_result["meal_plan"] is None
    assert isinstance(network_result["generation_time"], (int, float))

    message = network_result["error"]

    assert "MealPlanGenerator.generate" in message
    assert expected_type in message
    assert "recipe_chain.invoke" in message
    assert expected_suggestion in message
    assert "HTTP " not in message
    assert "PRIVATE_CONNECTION_MARKER" not in message

    print(message)

print("All network error handling checks passed.")

MealPlanGenerator.generate | APITimeoutError | recipe_chain.invoke | Request timed out. Check your connection and try again later.
MealPlanGenerator.generate | APIConnectionError | recipe_chain.invoke | Could not connect to the API. Check your network, proxy, and firewall settings.
All network error handling checks passed.


In [94]:
import httpx
from openai import InternalServerError
from unittest.mock import patch

# Construct a simulated server error without sending a request.
server_response = httpx.Response(
    status_code=500,
    request=httpx.Request(
        "POST",
        "https://api.openai.com/v1/chat/completions",
    ),
)

server_error = InternalServerError(
    message="Simulated server failure: PRIVATE_SERVER_MARKER",
    response=server_response,
    body={"error": {"message": "Simulated server failure"}},
)

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

with patch.object(
    chain_type,
    "invoke",
    side_effect=server_error,
) as invoke_mock:
    server_result = generator.generate(
        dietary_preference="omnivore",
        theme_requests="random",
        additional_preferences="none",
    )

    invoke_mock.assert_called_once()

assert server_result["success"] is False
assert server_result["meal_plan"] is None
assert isinstance(server_result["generation_time"], (int, float))

message = server_result["error"]

assert "MealPlanGenerator.generate" in message
assert "InternalServerError" in message
assert "recipe_chain.invoke" in message
assert "HTTP 500" in message
assert "check service availability" in message
assert "PRIVATE_SERVER_MARKER" not in message

print(message)
print("API status error handling check passed.")

MealPlanGenerator.generate | InternalServerError | recipe_chain.invoke | HTTP 500 | The API rejected the request. For 5xx errors, check service availability and try later; for other errors, check request parameters and model access.
API status error handling check passed.


In [96]:
import json
from types import SimpleNamespace
from unittest.mock import patch

# Prepare malformed JSON and determine its expected error location.
malformed_content = '{"recipes": [}'

try:
    json.loads(malformed_content)
except json.JSONDecodeError as error:
    expected_line = error.lineno
    expected_column = error.colno
    expected_position = error.pos
else:
    raise AssertionError("The test input must contain invalid JSON.")

# Return a fake model response without calling the API.
fake_response = SimpleNamespace(content=malformed_content)
chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

with patch.object(
    chain_type,
    "invoke",
    return_value=fake_response,
) as invoke_mock:
    json_result = generator.generate(
        dietary_preference="omnivore",
        theme_requests="random",
        additional_preferences="none",
    )

    invoke_mock.assert_called_once()

assert json_result["success"] is False
assert json_result["meal_plan"] is None
assert isinstance(json_result["generation_time"], (int, float))

message = json_result["error"]

assert "MealPlanGenerator.generate" in message
assert "JSONDecodeError" in message
assert "Response JSON parsing" in message
assert f"line {expected_line}," in message
assert f"column {expected_column}," in message
assert f"position {expected_position} |" in message
assert "Check JSON syntax" in message
assert malformed_content not in message

print(message)
print("JSON parsing error handling check passed.")

MealPlanGenerator.generate | JSONDecodeError | Response JSON parsing: line 1, column 14, position 13 | The model response contains invalid JSON. Check JSON syntax, quotation marks, commas, and brackets.
JSON parsing error handling check passed.


In [98]:
import json
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

# Load a saved valid plan without making an API request.
invalid_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

# Deliberately violate the seven-recipe requirement.
invalid_plan["recipes"] = invalid_plan["recipes"][:6]
assert len(invalid_plan["recipes"]) == 6

fake_response = SimpleNamespace(
    content=json.dumps(invalid_plan)
)

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

with patch.object(
    chain_type,
    "invoke",
    return_value=fake_response,
) as invoke_mock:
    validation_result = generator.generate(
        dietary_preference="omnivore",
        theme_requests="random",
        additional_preferences="none",
    )

    invoke_mock.assert_called_once()

assert validation_result["success"] is False
assert validation_result["meal_plan"] is None
assert isinstance(validation_result["generation_time"], (int, float))

message = validation_result["error"]

assert "MealPlanGenerator.generate" in message
assert "ValidationError" in message
assert "MealPlan schema" in message
assert "recipes: value_error" in message
assert "Check the listed fields" in message

print(message)
print("Meal-plan validation error handling check passed.")

MealPlanGenerator.generate | ValidationError | MealPlan schema: recipes: value_error | The response does not match the meal-plan requirements. Check the listed fields, required values, recipe count, ingredient count, and unique themes.
Meal-plan validation error handling check passed.


In [102]:
import json
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

# Load saved data without calling the API.
valid_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

# Create an independent invalid copy.
invalid_plan = json.loads(json.dumps(valid_plan))
invalid_plan["recipes"] = invalid_plan["recipes"][:6]

expected_keys = {
    "success",
    "meal_plan",
    "generation_time",
    "error",
}

contract_cases = [
    (
        "success",
        SimpleNamespace(content=json.dumps(valid_plan)),
        None,
        True,
    ),
    (
        "invalid JSON",
        SimpleNamespace(content='{"recipes": [}'),
        None,
        False,
    ),
    (
        "invalid schema",
        SimpleNamespace(content=json.dumps(invalid_plan)),
        None,
        False,
    ),
    (
        "unexpected failure",
        None,
        RuntimeError("PRIVATE_CONTRACT_MARKER"),
        False,
    ),
]

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

for case_name, fake_response, fake_error, expected_success in contract_cases:
    with patch.object(
        chain_type,
        "invoke",
        return_value=fake_response,
        side_effect=fake_error,
    ) as invoke_mock:
        result = generator.generate(
            dietary_preference="omnivore",
            theme_requests="random",
            additional_preferences="none",
        )

        invoke_mock.assert_called_once()

    assert set(result) == expected_keys, case_name
    assert result["success"] is expected_success, case_name
    assert type(result["generation_time"]) in (int, float), case_name
    assert result["generation_time"] >= 0, case_name

    if expected_success:
        assert result["meal_plan"] == valid_plan, case_name
        assert result["error"] is None, case_name
    else:
        assert result["meal_plan"] is None, case_name
        assert isinstance(result["error"], str), case_name
        assert result["error"].strip(), case_name
        assert "PRIVATE_CONTRACT_MARKER" not in result["error"], case_name

    print(f"{case_name}: result contract passed")

print("All result contract checks passed.")

success: result contract passed
invalid JSON: result contract passed
invalid schema: result contract passed
unexpected failure: result contract passed
All result contract checks passed.


In [101]:
import json
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

# Load a valid saved plan without calling the API.
saved_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)
plan_json = json.dumps(saved_plan)

response_cases = [
    ("json fence", f"```json\n{plan_json}\n```", True),
    ("unlabeled fence", f"```\n{plan_json}\n```", True),
    ("incomplete fence", f"```json\n{plan_json}", False),
    ("JSON list", "[]", False),
    ("fenced JSON list", "```json\n[]\n```", False),
    ("unlabeled fenced list", "```\n[]\n```", False),
    ("unsupported fence label", f"```python\n{plan_json}\n```", False),
    ("text before fence", f"Here is your plan:\n```json\n{plan_json}\n```", False),
    ("text after fence", f"```json\n{plan_json}\n```\nAdditional text", False),
]

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

for case_name, content, expected_success in response_cases:
    fake_response = SimpleNamespace(content=content)

    with patch.object(
        chain_type,
        "invoke",
        return_value=fake_response,
    ) as invoke_mock:
        result = generator.generate(
            dietary_preference="omnivore",
            theme_requests="random",
            additional_preferences="none",
        )

        invoke_mock.assert_called_once()

    assert result["success"] is expected_success, case_name

    if expected_success:
        assert result["meal_plan"] == saved_plan, case_name
        assert result["error"] is None, case_name
    else:
        assert result["meal_plan"] is None, case_name
        assert "MealPlanGenerator.generate" in result["error"], case_name

        if case_name == "text before fence":
            assert "JSONDecodeError" in result["error"], case_name
            assert "Response JSON parsing" in result["error"], case_name
            assert "Check JSON syntax" in result["error"], case_name
        else:
            assert "ValueError" in result["error"], case_name
            assert "Response processing" in result["error"], case_name
            assert (
                "Check that the response contains a JSON object"
                in result["error"]
            ), case_name

    print(f"{case_name}: passed")

print("All generation JSON format checks passed.")

json fence: passed
unlabeled fence: passed
incomplete fence: passed
JSON list: passed
fenced JSON list: passed
unlabeled fenced list: passed
unsupported fence label: passed
text before fence: passed
text after fence: passed
All generation JSON format checks passed.


In [109]:
import json
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

# Return a saved valid plan instead of calling the API.
request_test_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

fake_response = SimpleNamespace(
    content=json.dumps(request_test_plan)
)

input_cases = [
    (None, None, "random diverse cuisines", "none"),
    ("", "", "random diverse cuisines", "none"),
    ("   ", "   ", "random diverse cuisines", "none"),
    (" Italian ", " no nuts ", "Italian", "no nuts"),
]

chain_type = type(MEAL_PLAN_PROMPT | generator.llm)

for themes, preferences, expected_themes, expected_preferences in input_cases:
    with patch.object(
        chain_type,
        "invoke",
        return_value=fake_response,
    ) as invoke_mock:
        result = generator.generate(
            dietary_preference="omnivore",
            theme_requests=themes,
            additional_preferences=preferences,
        )

        invoke_mock.assert_called_once()

        forwarded_request = invoke_mock.call_args.args[0]

        assert forwarded_request == {
            "dietary_preference": "omnivore",
            "theme_requests": expected_themes,
            "additional_preferences": expected_preferences,
            "format_instructions": (
                generator.recipe_parser.get_format_instructions()
            ),
        }
        assert result["success"] is True

print("All generation request preparation checks passed.")

All generation request preparation checks passed.


In [171]:
def format_ingredient_list(
    ingredients: list[dict],
    separator: str = ", ",
) -> str:
    """Format ingredient quantities and names as a single line."""
    return separator.join(f"{item['quantity']} {item['name']}" for item in ingredients)

In [9]:
# Checking the list of two ingredients
sample_ingredients = [
    {"quantity": "2 cups", "name": "rice"},
    {"quantity": "1 tbsp", "name": "soy sauce"},
]

result = format_ingredient_list(sample_ingredients)

print(result)

assert result == "2 cups rice, 1 tbsp soy sauce"
assert format_ingredient_list([{"quantity": "3", "name": "eggs"}]) == "3 eggs"
assert format_ingredient_list([]) == ""
assert format_ingredient_list(
    sample_ingredients, separator="; "
) == "2 cups rice; 1 tbsp soy sauce"

print("All ingredient formatting tests passed.")

2 cups rice, 1 tbsp soy sauce
All ingredient formatting tests passed.


In [172]:
def normalize_optional_text(
    value: str | None,
    default: str,
) -> str:
    """Trim optional text and use the default when it is empty."""
    if value is None or value.strip() == "":
        return default
    return value.strip()

In [16]:
assert normalize_optional_text(None, "random") == "random"
assert normalize_optional_text("", "random") == "random"
assert normalize_optional_text("   ", "random") == "random"
assert normalize_optional_text("  Italian  ", "random") == "Italian"

# Let's check that the `default` parameter isn't hard-coded inside the function.
assert normalize_optional_text(None, "none") == "none"
assert normalize_optional_text(" no nuts ", "none") == "no nuts"

print("Normalized theme:", normalize_optional_text(" Italian ", "random"))
print("Empty preferences:", normalize_optional_text(None, "none"))
print("All optional text normalization tests passed.")

Normalized theme: Italian
Empty preferences: none
All optional text normalization tests passed.


## 6. Email Functionality

In [173]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients = format_ingredient_list(recipe["ingredients"])
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)

def build_meal_plan_message(
    sender: str,
    recipient: str,
    meal_plan: dict,
) -> MIMEMultipart:
    """Build an email message without connecting to an SMTP server."""
    message = MIMEMultipart()
    message["From"] = sender
    message["To"] = recipient
    message["Subject"] = "Your Weekly Meal Plan"

    body = format_meal_plan_email(meal_plan)
    message.attach(MIMEText(body, "plain", "utf-8"))

    return message

def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send meal plan via email."""
    missing_settings = []

    if not EMAIL_SENDER:
        missing_settings.append("EMAIL_SENDER")

    if not EMAIL_PASSWORD or not EMAIL_PASSWORD.strip():
        missing_settings.append("EMAIL_PASSWORD")

    if missing_settings:
        return {
            "success": False,
            "message": (
                "send_meal_plan_email | ConfigurationError | Email configuration | "
                f"Missing settings: {', '.join(missing_settings)}. "
                "Set them in .env or the environment, then rerun the configuration cell."
            ),
        }

    try:
        msg = build_meal_plan_message(
            sender=EMAIL_SENDER,
            recipient=recipient_email,
            meal_plan=meal_plan,
        )

        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)

        return {"success": True, "message": f"Email sent to {recipient_email}"}

    except smtplib.SMTPAuthenticationError as e:
        return {
            "success": False,
            "message": (
                "send_meal_plan_email | SMTPAuthenticationError | "
                f"SMTP login: smtp.gmail.com:587 | SMTP {e.smtp_code} | "
                "Email authentication failed. Check EMAIL_SENDER and "
                "the email provider's app-password requirements."
            ),
        }
    
    except smtplib.SMTPException as e:
        return {
            "success": False,
            "message": (
                f"send_meal_plan_email | {type(e).__name__} | "
                "SMTP delivery: smtp.gmail.com:587 | "
                "Email delivery failed. Check the SMTP server settings, "
                "recipient address, and provider restrictions."
            ),
        }

    except TimeoutError:
        return {
            "success": False,
            "message": (
                "send_meal_plan_email | TimeoutError | "
                "SMTP connection or delivery: smtp.gmail.com:587 | "
                "The network operation timed out. "
                "Check your connection and SMTP availability before retrying."
            ),
        }

    except OSError as e:
        return {
            "success": False,
            "message": (
                f"send_meal_plan_email | {type(e).__name__} | "
                "SMTP connection or delivery: smtp.gmail.com:587 | "
                "A network or operating-system error occurred. "
                "Check the network, DNS, firewall, and SMTP server address."
            ),
        }

    except Exception as e:
        return {
            "success": False,
            "message": (
                f"send_meal_plan_email | {type(e).__name__} | "
                "Email pipeline | An unexpected failure occurred. "
                "Check message construction and email delivery settings "
                "using an isolated offline test."
            ),
        }

In [130]:
from unittest.mock import patch

original_sender = EMAIL_SENDER
original_password = EMAIL_PASSWORD

try:
    test_cases = [
        ("", "", ["EMAIL_SENDER", "EMAIL_PASSWORD"]),
        ("", "fake-test-password", ["EMAIL_SENDER"]),
        ("sender@example.com", "", ["EMAIL_PASSWORD"]),
        ("sender@example.com", "   ", ["EMAIL_PASSWORD"]),
    ]

    with (
        patch.object(smtplib, "SMTP") as smtp_mock,
        patch("__main__.build_meal_plan_message") as builder_mock,
    ):
        for sender, password, expected_missing in test_cases:
            EMAIL_SENDER = sender
            EMAIL_PASSWORD = password

            result = send_meal_plan_email("recipient@example.com", {})

            assert result["success"] is False
            assert "send_meal_plan_email" in result["message"]
            assert "ConfigurationError" in result["message"]

            for setting in expected_missing:
                assert setting in result["message"]

            assert "fake-test-password" not in result["message"]
            smtp_mock.assert_not_called()
            builder_mock.assert_not_called()

            print(result["message"])

    print("All missing email configuration checks passed.")

finally:
    EMAIL_SENDER = original_sender
    EMAIL_PASSWORD = original_password

send_meal_plan_email | ConfigurationError | Email configuration | Missing settings: EMAIL_SENDER, EMAIL_PASSWORD. Set them in .env or the environment, then rerun the configuration cell.
send_meal_plan_email | ConfigurationError | Email configuration | Missing settings: EMAIL_SENDER. Set them in .env or the environment, then rerun the configuration cell.
send_meal_plan_email | ConfigurationError | Email configuration | Missing settings: EMAIL_PASSWORD. Set them in .env or the environment, then rerun the configuration cell.
send_meal_plan_email | ConfigurationError | Email configuration | Missing settings: EMAIL_PASSWORD. Set them in .env or the environment, then rerun the configuration cell.
All missing email configuration checks passed.


In [118]:
import json
from pathlib import Path
from unittest.mock import patch

# Load a saved plan without calling the API.
message_test_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

# Include non-ASCII text to verify UTF-8 encoding.
message_test_plan["recipes"][0]["dish_name"] = "Crème brûlée"

with patch.object(smtplib, "SMTP") as smtp_mock:
    message = build_meal_plan_message(
        sender="sender@example.com",
        recipient="recipient@example.com",
        meal_plan=message_test_plan,
    )

    smtp_mock.assert_not_called()

assert message["From"] == "sender@example.com"
assert message["To"] == "recipient@example.com"
assert message["Subject"] == "Your Weekly Meal Plan"

assert message.is_multipart()

parts = message.get_payload()
assert len(parts) == 1

text_part = parts[0]

assert text_part.get_content_type() == "text/plain"
assert text_part.get_content_charset() == "utf-8"

decoded_body = text_part.get_payload(decode=True).decode("utf-8")

assert decoded_body == format_meal_plan_email(message_test_plan)
assert "Crème brûlée" in decoded_body

print("Headers, body, and UTF-8 encoding checks passed.")
print("Message construction did not connect to SMTP.")

Headers, body, and UTF-8 encoding checks passed.
Message construction did not connect to SMTP.


In [128]:
import json
from pathlib import Path
from unittest.mock import patch

# Load saved data without calling the API.
delivery_test_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

original_sender = EMAIL_SENDER
original_password = EMAIL_PASSWORD

try:
    # Use artificial credentials only.
    EMAIL_SENDER = "sender@example.com"
    EMAIL_PASSWORD = "fake-test-password"

    with patch.object(smtplib, "SMTP") as smtp_mock:
        fake_server = smtp_mock.return_value.__enter__.return_value

        result = send_meal_plan_email(
            "recipient@example.com",
            delivery_test_plan,
        )

        smtp_mock.assert_called_once_with(
            "smtp.gmail.com", 587, timeout=20
        )
        fake_server.starttls.assert_called_once_with()
        fake_server.login.assert_called_once_with(
            "sender@example.com",
            "fake-test-password",
        )
        fake_server.send_message.assert_called_once()

        sent_message = fake_server.send_message.call_args.args[0]

        assert sent_message["From"] == "sender@example.com"
        assert sent_message["To"] == "recipient@example.com"
        assert sent_message["Subject"] == "Your Weekly Meal Plan"

        text_part = sent_message.get_payload()[0]
        sent_body = text_part.get_payload(decode=True).decode("utf-8")

        assert sent_body == format_meal_plan_email(delivery_test_plan)
        assert result["success"] is True
        assert "fake-test-password" not in result["message"]

    print("SMTP delivery integration checks passed.")
    print("No real email was sent.")

finally:
    EMAIL_SENDER = original_sender
    EMAIL_PASSWORD = original_password

SMTP delivery integration checks passed.
No real email was sent.


In [125]:
import json
from pathlib import Path
from unittest.mock import patch

# Load saved data without calling the API.
smtp_error_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

original_sender = EMAIL_SENDER
original_password = EMAIL_PASSWORD

try:
    EMAIL_SENDER = "sender@example.com"
    EMAIL_PASSWORD = "fake-test-password"

    with patch.object(smtplib, "SMTP") as smtp_mock:
        fake_server = smtp_mock.return_value.__enter__.return_value

        # Simulate an authentication failure without contacting a server.
        fake_server.login.side_effect = smtplib.SMTPAuthenticationError(
            535,
            b"PRIVATE_SMTP_MARKER",
        )

        result = send_meal_plan_email(
            "recipient@example.com",
            smtp_error_plan,
        )

        fake_server.login.assert_called_once_with(
            "sender@example.com",
            "fake-test-password",
        )
        fake_server.send_message.assert_not_called()

    assert result["success"] is False

    message = result["message"]

    assert "send_meal_plan_email" in message
    assert "SMTPAuthenticationError" in message
    assert "SMTP login: smtp.gmail.com:587" in message
    assert "SMTP 535" in message
    assert "Check EMAIL_SENDER" in message
    assert "PRIVATE_SMTP_MARKER" not in message
    assert "fake-test-password" not in message

    print(message)
    print("SMTP authentication error handling check passed.")

finally:
    EMAIL_SENDER = original_sender
    EMAIL_PASSWORD = original_password

send_meal_plan_email | SMTPAuthenticationError | SMTP login: smtp.gmail.com:587 | SMTP 535 | Email authentication failed. Check EMAIL_SENDER and the email provider's app-password requirements.
SMTP authentication error handling check passed.


In [124]:
import json
from pathlib import Path
from unittest.mock import patch

# Load saved data without calling the API.
smtp_failure_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

original_sender = EMAIL_SENDER
original_password = EMAIL_PASSWORD

try:
    EMAIL_SENDER = "sender@example.com"
    EMAIL_PASSWORD = "fake-test-password"

    failure_cases = [
        (
            smtplib.SMTPServerDisconnected("PRIVATE_SMTP_MARKER"),
            "SMTPServerDisconnected",
            "SMTP delivery: smtp.gmail.com:587",
            "Check the SMTP server settings",
        ),
        (
            RuntimeError("PRIVATE_RUNTIME_MARKER"),
            "RuntimeError",
            "Email pipeline",
            "Check message construction",
        ),
    ]

    for simulated_error, expected_type, expected_context, suggestion in failure_cases:
        with patch.object(smtplib, "SMTP") as smtp_mock:
            fake_server = smtp_mock.return_value.__enter__.return_value

            # Simulate failure at the actual delivery boundary.
            fake_server.send_message.side_effect = simulated_error

            result = send_meal_plan_email(
                "recipient@example.com",
                smtp_failure_plan,
            )

            fake_server.send_message.assert_called_once()

        assert result["success"] is False

        message = result["message"]

        assert "send_meal_plan_email" in message
        assert expected_type in message
        assert expected_context in message
        assert suggestion in message
        assert "PRIVATE_SMTP_MARKER" not in message
        assert "PRIVATE_RUNTIME_MARKER" not in message
        assert "fake-test-password" not in message

        print(f"{expected_type}: handling check passed")

    print("All SMTP and unexpected failure checks passed.")

finally:
    EMAIL_SENDER = original_sender
    EMAIL_PASSWORD = original_password

SMTPServerDisconnected: handling check passed
RuntimeError: handling check passed
All SMTP and unexpected failure checks passed.


In [129]:
from unittest.mock import patch

original_sender = EMAIL_SENDER
original_password = EMAIL_PASSWORD

try:
    EMAIL_SENDER = "sender@example.com"
    EMAIL_PASSWORD = "fake-test-password"

    network_cases = [
        (
            TimeoutError("PRIVATE_TIMEOUT_MARKER"),
            "TimeoutError",
            "Check your connection",
        ),
        (
            ConnectionRefusedError("PRIVATE_CONNECTION_MARKER"),
            "ConnectionRefusedError",
            "Check the network",
        ),
    ]

    for simulated_error, expected_type, suggestion in network_cases:
        with (
            patch(
                "__main__.build_meal_plan_message",
                return_value=object(),
            ),
            patch.object(
                smtplib,
                "SMTP",
                side_effect=simulated_error,
            ) as smtp_mock,
        ):
            result = send_meal_plan_email(
                "recipient@example.com",
                {},
            )

            smtp_mock.assert_called_once_with(
                "smtp.gmail.com", 587, timeout=20
            )

        assert result["success"] is False

        message = result["message"]

        assert "send_meal_plan_email" in message
        assert expected_type in message
        assert "smtp.gmail.com:587" in message
        assert suggestion in message
        assert "PRIVATE_TIMEOUT_MARKER" not in message
        assert "PRIVATE_CONNECTION_MARKER" not in message

        print(f"{expected_type}: handling check passed")

    print("All SMTP network error checks passed.")

finally:
    EMAIL_SENDER = original_sender
    EMAIL_PASSWORD = original_password

TimeoutError: handling check passed
ConnectionRefusedError: handling check passed
All SMTP network error checks passed.


In [11]:
import json
from pathlib import Path

saved_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

email_preview = format_meal_plan_email(saved_plan)

# Проверяем, что ингредиенты каждого рецепта присутствуют в тексте.
for recipe in saved_plan["recipes"]:
    expected_ingredients = ", ".join(
        f"{item['quantity']} {item['name']}"
        for item in recipe["ingredients"]
    )
    assert f"Ingredients: {expected_ingredients}" in email_preview

print(email_preview)
print("\nEmail ingredient formatting checks passed.")

WEEKLY MEAL PLAN (OMNIVORE)

SHOPPING LIST
--------------------
- 200g spaghetti
- 4 cloves garlic
- 50ml olive oil
- 1 tsp red pepper flakes
- 1/4 cup parsley
- 300g chicken breast
- 4 taco shells
- 1 cup lettuce
- 3 tomato
- 100g cheddar cheese
- 2 salmon fillets
- 50ml soy sauce
- 2 tbsp honey
- 1 inch ginger
- 2 green onions
- 300g chicken thighs
- 80g butter
- 200g tomato puree
- 100ml cream
- 1 tbsp garam masala
- 200g rice noodles
- 150g tofu
- 1 cup bean sprouts
- 2 tbsp peanut butter
- 1 lime
- 1 cucumber
- 100g feta cheese
- 1/2 cup olives
- 1/2 red onion
- 4 chicken drumsticks
- 100ml BBQ sauce
- 2 corn on the cob
- 4 potatoes

RECIPES

MONDAY - Italian
Spaghetti Aglio e Olio
Ingredients: 200g spaghetti, 4 cloves garlic, 50ml olive oil, 1 tsp red pepper flakes, 1/4 cup parsley
Instructions: Cook spaghetti. Sauté garlic in olive oil, add red pepper flakes. Toss with spaghetti and parsley.

TUESDAY - Mexican
Chicken Tacos
Ingredients: 300g chicken breast, 4 taco shells, 1 cup 

## 7. Validation

In [174]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Check a generation result using the shared meal-plan schema."""
    if not meal_plan_result.get("success"):
        return {
            "passed": False,
            "errors": [
                meal_plan_result.get("error") or "Meal-plan generation failed."
            ],
        }

    try:
        validate_meal_plan(meal_plan_result["meal_plan"])
    except ValidationError as error:
        field_errors = []

        for issue in error.errors():
            location = ".".join(str(part) for part in issue["loc"])
            location = location or "meal_plan"
            field_errors.append(
                "run_basic_validation | ValidationError | "
                f"MealPlan schema: {location}: {issue['type']} | "
                "Check the listed field against the meal-plan requirements."
            )

        return {
            "passed": False,
            "errors": field_errors,
        }

    return {
        "passed": True,
        "errors": [],
    }

In [83]:
import json
from copy import deepcopy
from pathlib import Path

# Load saved data without calling the API.
validation_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

valid_result = {
    "success": True,
    "meal_plan": validation_plan,
    "generation_time": 0,
    "error": None,
}

assert run_basic_validation(valid_result) == {
    "passed": True,
    "errors": [],
}

# Check each schema rule through the validation function.
invalid_cases = []

wrong_recipe_count = deepcopy(validation_plan)
wrong_recipe_count["recipes"] = wrong_recipe_count["recipes"][:6]
invalid_cases.append(("recipe count", wrong_recipe_count, "recipes"))

wrong_ingredient_count = deepcopy(validation_plan)
wrong_ingredient_count["recipes"][0]["ingredients"] = (
    wrong_ingredient_count["recipes"][0]["ingredients"][:4]
)
invalid_cases.append(
    ("ingredient count", wrong_ingredient_count, "recipes.0.ingredients")
)

duplicate_theme = deepcopy(validation_plan)
duplicate_theme["recipes"][1]["theme"] = duplicate_theme["recipes"][0]["theme"]
invalid_cases.append(("duplicate theme", duplicate_theme, "recipes"))

for case_name, invalid_plan, expected_location in invalid_cases:
    invalid_result = {
        "success": True,
        "meal_plan": invalid_plan,
        "generation_time": 0,
        "error": None,
    }

    checked = run_basic_validation(invalid_result)

    assert checked["passed"] is False, case_name
    assert checked["errors"], case_name
    assert any(
        expected_location in message
        for message in checked["errors"]
    ), case_name

    print(f"{case_name}: rejected")

# Preserve an existing generation failure.
failed_result = {
    "success": False,
    "meal_plan": None,
    "generation_time": 0,
    "error": "Simulated generation failure",
}

assert run_basic_validation(failed_result) == {
    "passed": False,
    "errors": ["Simulated generation failure"],
}

print("All shared validation checks passed.")

recipe count: rejected
ingredient count: rejected
duplicate theme: rejected
All shared validation checks passed.


In [ ]:
# Baseline: single manual generation and evidence export
from pathlib import Path
import json

# One input for the baseline.
baseline_input = {
    "dietary_preference": "omnivore",
    "theme_requests": "random",
    "additional_preferences": "none",
}

# Attention: this line makes one real request to the API.
baseline_result = generator.generate(**baseline_input)

# Check the generated result.
baseline_validation = run_basic_validation(baseline_result)

print("Generation successful:", baseline_result["success"])
print("Validation:", baseline_validation)

if baseline_result["success"]:
    plan = baseline_result["meal_plan"]

    print("Recipes:", len(plan["recipes"]))
    print(
        "Ingredients per recipe:",
        [len(recipe["ingredients"]) for recipe in plan["recipes"]],
    )

    # Check the generated result.
    evidence_dir = Path("evidence")
    evidence_dir.mkdir(exist_ok=True)

    (evidence_dir / "baseline_input.json").write_text(
        json.dumps(baseline_input, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    (evidence_dir / "baseline_output.json").write_text(
        json.dumps(plan, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    # Show the meal plan in the notebook output.
    print(json.dumps(plan, ensure_ascii=False, indent=2))
    print("Saved baseline files to:", evidence_dir.resolve())
else:
    print("Generation error:", baseline_result.get("error"))

Generation successful: True
Validation: {'passed': True, 'errors': []}
Recipes: 7
Ingredients per recipe: [5, 5, 5, 5, 5, 5, 5]
{
  "dietary_preference": "omnivore",
  "recipes": [
    {
      "day": "Monday",
      "theme": "Italian",
      "dish_name": "Spaghetti Aglio e Olio",
      "ingredients": [
        {
          "name": "spaghetti",
          "quantity": "200g"
        },
        {
          "name": "garlic",
          "quantity": "4 cloves"
        },
        {
          "name": "olive oil",
          "quantity": "50ml"
        },
        {
          "name": "red pepper flakes",
          "quantity": "1 tsp"
        },
        {
          "name": "parsley",
          "quantity": "1/4 cup"
        }
      ],
      "instructions": "Cook spaghetti. Sauté garlic in olive oil, add red pepper flakes. Toss with spaghetti and parsley."
    },
    {
      "day": "Tuesday",
      "theme": "Mexican",
      "dish_name": "Chicken Tacos",
      "ingredients": [
        {
          "name":

## 8. Gradio Interface

In [113]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = format_ingredient_list(recipe["ingredients"])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Display a generated plan and update the session's email state."""
    output = "Generating your meal plan..."

    # Clear the previous plan when a new generation starts.
    yield output, None

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=normalize_optional_text(theme_requests, "random"),
        additional_preferences=normalize_optional_text(additional_prefs, "none"),
    )

    if not result["success"]:
        yield f"Error: {result['error']}", None
        return

    validation = run_basic_validation(result)

    if not validation["passed"]:
        errors = "; ".join(validation["errors"])
        yield f"Validation failed: {errors}", None
        return

    meal_plan = result["meal_plan"]

    output = (
        f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    )
    output += format_shopping_list(meal_plan) + "\n\n---\n\n"
    output += "## Recipes\n\n"
    yield output, None

    for recipe in meal_plan["recipes"]:
        output += format_single_recipe(recipe) + "\n"
        yield output, None

    output += (
        f"---\n*Generated in {result['generation_time']}s | "
        "All checks passed*"
    )

    # Make the plan available for email after rendering completes.
    yield output, meal_plan


def send_email_ui(email, meal_plan):
    """Send the displayed meal plan without generating a new one."""
    if not email or "@" not in email:
        return "Enter a valid email address."

    if meal_plan is None:
        return "Generate a valid meal plan before sending email."

    email_result = send_meal_plan_email(email, meal_plan)
    return email_result["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    current_plan = gr.State(value=None)

    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display, current_plan],
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, current_plan],
        outputs=[email_status],
    )

In [114]:
from unittest.mock import patch

sample_email_plan = {
    "dietary_preference": "omnivore",
    "recipes": [],
    "shopping_list": [],
}

# Replace email delivery and block any unexpected generation.
with (
    patch(
        "__main__.send_meal_plan_email",
        return_value={"success": True, "message": "Simulated email sent"},
    ) as sender_mock,
    patch.object(
        generator,
        "generate",
        side_effect=AssertionError("Email must not generate a new plan."),
    ) as generation_mock,
):
    message = send_email_ui("recipient@example.com", sample_email_plan)

    assert message == "Simulated email sent"
    sender_mock.assert_called_once_with(
        "recipient@example.com",
        sample_email_plan,
    )
    assert sender_mock.call_args.args[1] is sample_email_plan
    generation_mock.assert_not_called()

    sender_mock.reset_mock()

    message = send_email_ui("recipient@example.com", None)

    assert message == "Generate a valid meal plan before sending email."
    sender_mock.assert_not_called()
    generation_mock.assert_not_called()

print("Email callback plan forwarding checks passed.")

Email callback plan forwarding checks passed.


In [14]:
# Manual test data: No file or API is required
sample_recipe = {
    "day": "Monday",
    "theme": "Italian",
    "dish_name": "Tomato Pasta",
    "ingredients": [
        {"quantity": "200 g", "name": "pasta"},
        {"quantity": "2", "name": "tomatoes"},
    ],
}

actual_output = format_single_recipe(sample_recipe)

expected_output = (
    "**Monday** | Italian | *Tomato Pasta*\n"
    "200 g pasta, 2 tomatoes\n"
)

assert actual_output == expected_output

print(actual_output)
print("Recipe UI formatting check passed.")

**Monday** | Italian | *Tomato Pasta*
200 g pasta, 2 tomatoes

Recipe UI formatting check passed.


In [112]:
class RecordingGenerator:
    """Record generation inputs without calling an API."""

    def __init__(self):
        self.calls = []

    def generate(self, **kwargs):
        self.calls.append(kwargs)

        return {
            "success": False,
            "meal_plan": None,
            "generation_time": 0,
            "error": "Intentional offline test stop",
        }


original_generator = generator
fake_generator = RecordingGenerator()

try:
    generator = fake_generator

    test_cases = [
        (None, None, "random", "none"),
        ("", "", "random", "none"),
        ("   ", "   ", "random", "none"),
        (" Italian ", " no nuts ", "Italian", "no nuts"),
    ]

    for themes, preferences, expected_themes, expected_preferences in test_cases:
        fake_generator.calls.clear()

        list(generate_plan_streaming("omnivore", themes, preferences))

        assert fake_generator.calls == [
            {
                "dietary_preference": "omnivore",
                "theme_requests": expected_themes,
                "additional_preferences": expected_preferences,
            }
        ]

    print("Generation callback normalization checks passed.")

finally:
    generator = original_generator

Generation callback normalization checks passed.


In [115]:
import json
from copy import deepcopy
from pathlib import Path
from unittest.mock import patch

# Load a valid plan without calling the API.
state_test_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

successful_result = {
    "success": True,
    "meal_plan": deepcopy(state_test_plan),
    "generation_time": 0.25,
    "error": None,
}

failed_result = {
    "success": False,
    "meal_plan": None,
    "generation_time": 0,
    "error": "Simulated generation failure",
}

# Check the successful generation lifecycle.
with patch.object(
    generator,
    "generate",
    return_value=successful_result,
) as generation_mock:
    updates = list(
        generate_plan_streaming("omnivore", "random", "none")
    )

    generation_mock.assert_called_once()

assert len(updates) >= 2
assert all(isinstance(update, tuple) and len(update) == 2 for update in updates)

# No plan is available for email until rendering finishes.
assert all(plan is None for _, plan in updates[:-1])

final_display, final_plan = updates[-1]

assert final_plan == state_test_plan
assert "All checks passed" in final_display

for recipe in final_plan["recipes"]:
    assert format_single_recipe(recipe) in final_display

print("Successful generation state checks passed.")

# Simulate a new generation after a previously successful one.
simulated_state = final_plan

with patch.object(
    generator,
    "generate",
    return_value=failed_result,
) as generation_mock:
    failed_updates = iter(
        generate_plan_streaming("omnivore", "random", "none")
    )

    first_display, simulated_state = next(failed_updates)

    assert first_display == "Generating your meal plan..."
    assert simulated_state is None

    for display, simulated_state in failed_updates:
        assert simulated_state is None

    generation_mock.assert_called_once()

assert "Simulated generation failure" in display

print("Failed generation cleared the previous plan.")
print("All generation state lifecycle checks passed.")

Successful generation state checks passed.
Failed generation cleared the previous plan.
All generation state lifecycle checks passed.


In [116]:
import json
from copy import deepcopy
from pathlib import Path
from unittest.mock import patch

# Load saved data without calling the API.
flow_plan = json.loads(
    Path("evidence/baseline_output.json").read_text(encoding="utf-8")
)

flow_result = {
    "success": True,
    "meal_plan": deepcopy(flow_plan),
    "generation_time": 0.25,
    "error": None,
}

with (
    patch.object(
        generator,
        "generate",
        return_value=flow_result,
    ) as generation_mock,
    patch(
        "__main__.send_meal_plan_email",
        return_value={
            "success": True,
            "message": "Simulated email sent",
        },
    ) as sender_mock,
):
    # Generate and collect the final display and session-state values.
    updates = list(
        generate_plan_streaming("omnivore", "random", "none")
    )
    displayed_text, displayed_plan = updates[-1]

    generation_mock.assert_called_once()

    assert displayed_plan == flow_plan
    assert "All checks passed" in displayed_text

    # Send the exact plan returned by the generation callback.
    email_message = send_email_ui(
        "recipient@example.com",
        displayed_plan,
    )

    assert email_message == "Simulated email sent"

    sender_mock.assert_called_once_with(
        "recipient@example.com",
        displayed_plan,
    )
    assert sender_mock.call_args.args[1] is displayed_plan

    # Sending email must not trigger another generation.
    generation_mock.assert_called_once()

print("Generate-to-email flow check passed.")
print("Generation calls: 1")
print("Email received the displayed plan.")

Generate-to-email flow check passed.
Generation calls: 1
Email received the displayed plan.


## 9. Launch Application

In [10]:
# Launch the Gradio app
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://16c83d37c44e2cf70f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [175]:
import json
from pathlib import Path


def load_test_cases(file_path: str | Path) -> list:
    """Load test requests from a UTF-8 JSON file."""
    path = Path(file_path)

    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

In [176]:
from pydantic import TypeAdapter

request_list_adapter = TypeAdapter(list[MealPlanRequest])


def validate_test_cases(data: object) -> list[MealPlanRequest]:
    """Validate a collection of meal-plan requests."""
    return request_list_adapter.validate_python(data)


def load_validated_test_cases(
    file_path: str | Path,
) -> list[MealPlanRequest]:
    """Load test requests and validate their structure."""
    data = load_test_cases(file_path)
    return validate_test_cases(data)

In [177]:
def read_test_cases_result(file_path: str | Path) -> dict:
    """Load validated requests and describe expected input failures."""
    path = Path(file_path)

    try:
        cases = load_validated_test_cases(path)

    except FileNotFoundError:
        message = (
            "read_test_cases_result | FileNotFoundError | "
            f"File: {path} | "
            "The input file was not found. "
            "Check the file path and the notebook working directory."
        )

    except json.JSONDecodeError as error:
        message = (
            "read_test_cases_result | JSONDecodeError | "
            f"File: {path}, line {error.lineno}, "
            f"column {error.colno}, position {error.pos} | "
            "The input file contains invalid JSON. "
            "Check quotation marks, commas, and brackets."
        )

    except ValidationError as error:
        locations = []

        for issue in error.errors():
            location = ".".join(str(part) for part in issue["loc"])
            location = location or "request_list"
            locations.append(f"{location}: {issue['type']}")

        message = (
            "read_test_cases_result | ValidationError | "
            f"File: {path}, fields: {'; '.join(locations)} | "
            "The input does not match the request schema. "
            "Check the listed fields and allowed dietary preferences."
        )

    else:
        return {
            "success": True,
            "cases": cases,
            "error": None,
        }

    return {
        "success": False,
        "cases": None,
        "error": message,
    }

In [158]:
from pathlib import Path

# Verify successful loading.
file_result = read_test_cases_result(
    "tests/fixtures/test_cases.json"
)

assert file_result["success"] is True
assert len(file_result["cases"]) == 4
assert file_result["error"] is None

# Verify actionable messages for actual file failures.
file_failure_cases = [
    (
        "tests/fixtures/missing_test_cases.json",
        "FileNotFoundError",
        "Check the file path",
        None,
    ),
    (
        "tests/fixtures/malformed.json",
        "JSONDecodeError",
        "Check quotation marks",
        "line ",
    ),
    (
        "tests/fixtures/invalid_requests.json",
        "ValidationError",
        "Check the listed fields",
        "1.dietary_preference",
    ),
]

for path, expected_type, suggestion, expected_location in file_failure_cases:
    result = read_test_cases_result(path)

    assert set(result) == {"success", "cases", "error"}
    assert result["success"] is False
    assert result["cases"] is None

    message = result["error"]

    assert "read_test_cases_result" in message
    assert expected_type in message
    assert f"File: {Path(path)}" in message
    assert suggestion in message

    if expected_location is not None:
        assert expected_location in message

    if expected_type == "JSONDecodeError":
        assert "column " in message
        assert "position " in message

    print(message)
    print()

print("All file error message checks passed.")

read_test_cases_result | FileNotFoundError | File: tests\fixtures\missing_test_cases.json | The input file was not found. Check the file path and the notebook working directory.

read_test_cases_result | JSONDecodeError | File: tests\fixtures\malformed.json, line 4, column 3, position 46 | The input file contains invalid JSON. Check quotation marks, commas, and brackets.

read_test_cases_result | ValidationError | File: tests\fixtures\invalid_requests.json, fields: 1.dietary_preference: enum | The input does not match the request schema. Check the listed fields and allowed dietary preferences.

All file error message checks passed.


In [151]:
loaded_cases = load_test_cases("tests/fixtures/test_cases.json")

assert isinstance(loaded_cases, list)
assert len(loaded_cases) == 4

assert [
    case["dietary_preference"]
    for case in loaded_cases
] == [
    "omnivore",
    "vegetarian",
    "vegan",
    "keto",
]

assert loaded_cases[1]["additional_preferences"] == "no mushrooms"

print("Loaded test cases:", len(loaded_cases))
print("Test case file loading check passed.")

Loaded test cases: 4
Test case file loading check passed.


In [152]:
from pydantic import ValidationError

# Validate all requests loaded from the fixture file.
request_cases = load_test_cases("tests/fixtures/test_cases.json")

for case in request_cases:
    validated_request = MealPlanRequest.model_validate(case)

    assert isinstance(
        validated_request.dietary_preference,
        DietaryPreference,
    )

# Check defaults for omitted optional fields.
minimal_request = MealPlanRequest.model_validate(
    {"dietary_preference": "vegan"}
)

assert minimal_request.theme_requests == "random"
assert minimal_request.additional_preferences == "none"

# Check explicitly empty optional values.
nullable_request = MealPlanRequest.model_validate(
    {
        "dietary_preference": "omnivore",
        "theme_requests": None,
        "additional_preferences": None,
    }
)

assert nullable_request.theme_requests is None
assert nullable_request.additional_preferences is None

# Reject an unsupported dietary preference.
try:
    MealPlanRequest.model_validate(
        {"dietary_preference": "unsupported-diet"}
    )
except ValidationError as error:
    assert any(
        issue["loc"] == ("dietary_preference",)
        for issue in error.errors()
    )
else:
    raise AssertionError("An unsupported diet was unexpectedly accepted.")

# Reject a non-string theme value.
try:
    MealPlanRequest.model_validate(
        {
            "dietary_preference": "omnivore",
            "theme_requests": ["Italian", "Mexican"],
        }
    )
except ValidationError as error:
    assert any(
        issue["loc"] == ("theme_requests",)
        for issue in error.errors()
    )
else:
    raise AssertionError("A list of themes was unexpectedly accepted.")

print("All request schema checks passed.")

All request schema checks passed.


In [136]:
from pydantic import ValidationError

# Verify loading and validation through the combined entry point.
validated_cases = load_validated_test_cases(
    "tests/fixtures/test_cases.json"
)

assert len(validated_cases) == 4
assert all(
    isinstance(case, MealPlanRequest)
    for case in validated_cases
)

assert validated_cases[0].dietary_preference.value == "omnivore"
assert validated_cases[1].additional_preferences == "no mushrooms"

# Verify that errors identify the request index and field.
invalid_cases = [
    {"dietary_preference": "omnivore"},
    {"dietary_preference": "unsupported-diet"},
]

try:
    validate_test_cases(invalid_cases)
except ValidationError as error:
    assert any(
        issue["loc"] == (1, "dietary_preference")
        for issue in error.errors()
    )
else:
    raise AssertionError("An invalid request was unexpectedly accepted.")

# Reject a JSON object where a list of requests is required.
try:
    validate_test_cases({"dietary_preference": "vegan"})
except ValidationError:
    pass
else:
    raise AssertionError("A non-list input was unexpectedly accepted.")

print("All validated test case loading checks passed.")

All validated test case loading checks passed.


In [137]:
import json
from pathlib import Path
from pydantic import ValidationError

# Verify a genuinely missing file.
missing_path = Path("tests/fixtures/missing_test_cases.json")
assert not missing_path.exists(), "Choose a filename that does not exist."

try:
    load_validated_test_cases(missing_path)
except FileNotFoundError as error:
    assert Path(error.filename) == missing_path
    print("Missing file: correctly rejected.")
else:
    raise AssertionError("A missing file was unexpectedly accepted.")

# Verify malformed JSON read from an actual file.
try:
    load_validated_test_cases("tests/fixtures/malformed.json")
except json.JSONDecodeError as error:
    assert error.lineno > 0
    assert error.colno > 0
    print(
        f"Malformed JSON: correctly rejected "
        f"at line {error.lineno}, column {error.colno}."
    )
else:
    raise AssertionError("Malformed JSON was unexpectedly accepted.")

# Verify schema errors in a syntactically valid JSON file.
try:
    load_validated_test_cases("tests/fixtures/invalid_requests.json")
except ValidationError as error:
    assert any(
        issue["loc"] == (1, "dietary_preference")
        for issue in error.errors()
    )
    print("Invalid request: correctly rejected at 1.dietary_preference.")
else:
    raise AssertionError("An invalid request was unexpectedly accepted.")

print("All file failure checks passed.")

Missing file: correctly rejected.
Malformed JSON: correctly rejected at line 4, column 3.
Invalid request: correctly rejected at 1.dietary_preference.
All file failure checks passed.


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [178]:
def run_test_cases_from_file(
    file_path: str | Path,
    *,
    allow_live_api: bool = False,
) -> dict:
    """Validate file-based requests and optionally run live generation."""
    loaded = read_test_cases_result(file_path)

    if not loaded["success"]:
        print(loaded["error"])
        return {
            "success": False,
            "mode": "input_error",
            "results": [],
            "error": loaded["error"],
        }

    cases = loaded["cases"]

    if not allow_live_api:
        print(f"Validated requests: {len(cases)}")
        print("Validation-only mode: no API calls were made.")

        return {
            "success": True,
            "mode": "validation_only",
            "results": [],
            "error": None,
        }

    results = []

    for case in cases:
        result = generator.generate(
            dietary_preference=case.dietary_preference.value,
            theme_requests=case.theme_requests,
            additional_preferences=case.additional_preferences,
        )

        validation = run_basic_validation(result)

        results.append({
            "dietary_preference": case.dietary_preference.value,
            "generation": result,
            "validation": validation,
        })

        status = "PASS" if validation["passed"] else "FAIL"
        print(f"{case.dietary_preference.value}: {status}")

    return {
        "success": all(
            item["validation"]["passed"]
            for item in results
        ),
        "mode": "live_api",
        "results": results,
        "error": None,
    }

In [160]:
from unittest.mock import patch

# Block generation to verify that the default mode stays offline.
with patch.object(
    generator,
    "generate",
    side_effect=AssertionError("Unexpected API generation attempt."),
) as generation_mock:
    runner_result = run_test_cases_from_file(
        "tests/fixtures/test_cases.json"
    )

    assert runner_result == {
        "success": True,
        "mode": "validation_only",
        "results": [],
        "error": None,
    }

    generation_mock.assert_not_called()

    invalid_result = run_test_cases_from_file(
        "tests/fixtures/invalid_requests.json"
    )

    assert invalid_result["success"] is False
    assert invalid_result["mode"] == "input_error"
    assert "ValidationError" in invalid_result["error"]

    generation_mock.assert_not_called()

print("File-based runner offline checks passed.")

Validated requests: 4
Validation-only mode: no API calls were made.
read_test_cases_result | ValidationError | File: tests\fixtures\invalid_requests.json, fields: 1.dietary_preference: enum | The input does not match the request schema. Check the listed fields and allowed dietary preferences.
File-based runner offline checks passed.


In [179]:
# Final live run: input, execution trace, and matching output.
import json
from datetime import datetime
from pathlib import Path
from uuid import uuid4

evidence_dir = Path("evidence")
evidence_dir.mkdir(exist_ok=True)

run_id = uuid4().hex
trace_lines = []


def record_step(message: str) -> None:
    """Record a timestamped execution event."""
    timestamp = datetime.now().astimezone().isoformat(timespec="seconds")
    line = f"{timestamp} | {run_id} | {message}"
    trace_lines.append(line)
    print(line)


final_input = {
    "dietary_preference": "omnivore",
    "theme_requests": "random",
    "additional_preferences": "none",
}

input_path = evidence_dir / "refactored_input.json"
output_path = evidence_dir / "refactored_output.json"
trace_path = evidence_dir / "execution_trace.txt"

input_path.write_text(
    json.dumps(final_input, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Remove a previous output so a failed run cannot leave misleading evidence.
if output_path.exists():
    output_path.unlink()

try:
    record_step(f"Input saved: {input_path.as_posix()}")
    record_step(f"Live generation started; model={OPENAI_MODEL}")

    final_result = generator.generate(**final_input)

    record_step(
        f"Generation completed; success={final_result['success']}"
    )

    final_validation = run_basic_validation(final_result)

    record_step(
        f"Schema validation completed; passed={final_validation['passed']}"
    )

    if not final_result["success"] or not final_validation["passed"]:
        raise RuntimeError(
            "Final run failed. Inspect final_result and final_validation."
        )

    output_path.write_text(
        json.dumps(
            final_result["meal_plan"],
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    record_step(f"Validated output saved: {output_path.as_posix()}")

    print(json.dumps(
        final_result["meal_plan"],
        ensure_ascii=False,
        indent=2,
    ))
    print("Final live run evidence saved.")

finally:
    trace_path.write_text(
        "\n".join(trace_lines) + "\n",
        encoding="utf-8",
    )

2026-10-19T22:56:48+02:00 | 5645fc29a2f04751ae6a78fc02b24bde | Input saved: evidence/refactored_input.json
2026-10-19T22:56:48+02:00 | 5645fc29a2f04751ae6a78fc02b24bde | Live generation started; model=gpt-4o-mini
2026-10-19T22:57:09+02:00 | 5645fc29a2f04751ae6a78fc02b24bde | Generation completed; success=True
2026-10-19T22:57:09+02:00 | 5645fc29a2f04751ae6a78fc02b24bde | Schema validation completed; passed=True
2026-10-19T22:57:09+02:00 | 5645fc29a2f04751ae6a78fc02b24bde | Validated output saved: evidence/refactored_output.json
{
  "dietary_preference": "omnivore",
  "recipes": [
    {
      "day": "Monday",
      "theme": "Italian",
      "dish_name": "Spaghetti Carbonara",
      "ingredients": [
        {
          "name": "spaghetti",
          "quantity": "200g"
        },
        {
          "name": "pancetta",
          "quantity": "100g"
        },
        {
          "name": "parmesan cheese",
          "quantity": "50g"
        },
        {
          "name": "eggs",
          